In [1]:
%matplotlib qt


In [2]:
import yt
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt

from mpl_toolkits.axes_grid1 import make_axes_locatable


jetcmap = plt.cm.get_cmap("jet", 9) #generate a jet map with 10 values "rainbow", "jet", YlOrRd
jet_vals = jetcmap(np.arange(9)) #extract those values as an array 
jet_vals[0] = [1.0, 1, 1.0, 1] #change the first value 
jet_vals[8] = [0.0, 0, 0.0, 1] #change the first value 
newcmap = mpl.colors.LinearSegmentedColormap.from_list("mine", jet_vals) 

from matplotlib import font_manager

font_dirs = ['/Users/yao/Documents/Calibri and Cambria Fonts/']
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

# set font
plt.rcParams['font.family'] = 'Calibri'

plt.rc('text', usetex=False)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)
plt.rc('axes', labelsize=20)
plt.rc('axes', titlesize=22)
plt.rc('legend', fontsize=14)

/var/folders/2t/97rc3fl92tg15k2l_4sk5hsh0000gn/T/ipykernel_10817/744338052.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  jetcmap = plt.cm.get_cmap("jet", 9) #generate a jet map with 10 values "rainbow", "jet", YlOrRd


In [92]:
filedir = [
        '/Users/yao/Documents/Data/LULI2000/neutron_Vincent/vincent_t5_2um/',
          ] 

filename = 'GEKKO_hdf5_chk_0000'

In [93]:
data_yt = yt.load(filedir[0] + filename)

yt : [INFO     ] 2026-03-13 11:46:47,712 Particle file found: GEKKO_hdf5_chk_0000


yt : [WARNING  ] 2026-03-13 11:46:47,718 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 11:46:47,736 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-13 11:46:47,737 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 11:46:47,737 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 11:46:47,737 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 11:46:47,738 Parameters: cosmological_simulation   = 0


In [94]:
data_yt_map = data_yt.covering_grid(
        level=0, left_edge=[0, 0.0, 0.0], dims=data_yt.domain_dimensions
    )

In [95]:
dens = data_yt_map['dens'][:,:,0].T
targ = data_yt_map['targ'][:,:,0].T
# dens = data_yt_map['cham'][:,:,0].T

In [96]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --------------------------------------------------
# Geometry of the integration region
# --------------------------------------------------
z_top = 0.803
z_bottom = 0.75

theta_deg = 25.0
theta = np.deg2rad(theta_deg)

# angle measured from the vertical
r0 = 0.005 + 0.06 * np.tan(theta)
z0 = z_top

# --------------------------------------------------
# Coordinates
# --------------------------------------------------
# rr   : 1D radial array in cm
# zz   : 1D axial array in cm
# dens : 2D mass density array in g/cm^3
#
# IMPORTANT:
# dens must be the VANADIUM mass density if you want the Vanadium ion number directly.
# If dens is the total mass density, see note below.
rr = np.linspace(0, 1, dens.shape[0])  # example radial coordinates
zz = np.linspace(0, 1, dens.shape[1])  # example axial coordinates
R, Z = np.meshgrid(rr, zz)

# right boundary of the mask
rmax = r0 + (z0 - Z) * np.tan(theta)

# mask
mask = (
    (Z >= z_bottom) &
    (Z <= z_top) &
    (R >= 0.0) &
    (R <= rmax)
)

# lines for plotting
z_line = np.linspace(z_bottom, z_top, 400)
r_line = r0 + (z0 - z_line) * np.tan(theta)
r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

# plot uses log10, but integration will use dens itself
f1 = ax.imshow(
    np.log10(dens),
    cmap=newcmap,
    origin='lower',
    extent=[rr.min(), rr.max(), zz.min(), zz.max()],
    vmin=-6, vmax=1,
    aspect='equal',
)

# soft shading
ax.contourf(
    R, Z, mask.astype(float),
    levels=[0.5, 1.5],
    colors=['magenta'],
    alpha=0.2
)

# mask contour
# ax.contour(
#     R, Z, mask.astype(float),
#     levels=[0.5],
#     colors='magenta',
#     linewidths=2.0
# )

# boundary lines
ax.plot(r_line, z_line, '--', color='white', lw=2.5, label='integration boundary')
ax.plot([0, r_bottom], [z_bottom, z_bottom], '-', color='white', lw=2.5)
ax.plot([0, r0], [z_top, z_top], '-', color='white', lw=2.5)
ax.plot(r0, z0, 'wo', ms=7)
ax.text(r0 + 0.002, z0 + 0.001, r'$(r_0,z_0)$', color='white', fontsize=14)

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')

fig.colorbar(f1, cax=cax, orientation='vertical')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

# --------------------------------------------------
# Integration: Vanadium ion number
# --------------------------------------------------
# dens is in g/cm^3
# Need number density:
#   n_V = rho_V / (A_V * m_u)
#
# A_V: atomic mass of Vanadium
# m_u: atomic mass unit in g

A_V = 50.9415
m_u = 1.66053906660e-24   # g

# If dens is already VANADIUM mass density:
rho_V = dens * targ    # g/cm^3

n_V = rho_V / (A_V * m_u)   # cm^-3

# cell sizes
dr = np.gradient(rr)
dz = np.gradient(zz)

DR, DZ = np.meshgrid(dr, dz)

# axisymmetric cell volume
dV = 2.0 * np.pi * R * DR * DZ   # cm^3

# total Vanadium ion number
N_V = np.sum(n_V[mask] * dV[mask])

# optional: total Vanadium mass in the region
M_V = np.sum(rho_V[mask] * dV[mask])   # g

print(f"theta = {theta_deg:.1f} deg")
print(f"r0 = {r0:.6f} cm")
print(f"z_top = {z_top:.6f} cm")
print(f"z_bottom = {z_bottom:.6f} cm")
print(f"Total Vanadium mass in mask = {M_V:.6e} g")
print(f"Total Vanadium ion number in mask = {N_V:.6e}")

theta = 25.0 deg
r0 = 0.032978 cm
z_top = 0.803000 cm
z_bottom = 0.750000 cm
Total Vanadium mass in mask = 4.384824e-06 code_mass/code_length**3 g
Total Vanadium ion number in mask = 5.183598e+16 code_mass/code_length**3


In [75]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

# --------------------------------
# Plotting field
# --------------------------------
logdens = np.log10(dens)

# Values below this threshold will be shown in white
threshold = -5.5

# Mask very small densities
logdens_masked = np.ma.masked_where(logdens < threshold, logdens)

# Recommended colormap
cmap = plt.cm.inferno.copy()
cmap.set_bad(color='white')

f1 = ax.imshow(
    logdens_masked,
    cmap=cmap,
    origin='lower',
    extent=[rr.min(), rr.max(), zz.min(), zz.max()],
    vmin=-6,
    vmax=1,
    aspect='equal',
)

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')
# ax.set_title('Time = {:.1f} ns'.format(time_now))

fig.colorbar(f1, cax=cax, orientation='vertical')

fig.tight_layout()
plt.show()

In [73]:
fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

logdens = np.log10(dens)

threshold = -6.5   # choose what you want

masked = np.ma.masked_where(logdens < threshold, logdens)

newcmap = plt.cm.jet.copy()
newcmap.set_bad(color='white')

f1 = ax.imshow(
        # np.log10(dens),
        dens,
                 cmap=newcmap,
                 origin='lower',
                 extent=[0,1,0,1],
                #  vmin=-6,vmax=1,
                 aspect='equal',
                 )

# ax.set_xlim(xx0,xx1)
# ax.set_ylim(yy0,yy1)
ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')
# ax.set_title('Time = {:.1f} ns'.format(time_now))

fig.colorbar(f1, cax=cax, orientation='vertical')

# fig.set_size_inches(width, height)
fig.tight_layout()

In [ ]:
xmin = data_yt_map.LeftEdge[0]
xmax = data_yt_map.RightEdge[0]
ymin = data_yt_map.LeftEdge[1]
ymax = data_yt_map.RightEdge[1]
xx = np.linspace(xmin, xmax, dens.shape[0])
yy = np.linspace(ymin, ymax, dens.shape[1])
XX, YY = np.meshgrid(xx, yy)


In [33]:
import numpy as np
import matplotlib.pyplot as plt

# parameters
z_top = 0.80
z_bottom = 0.735

theta_deg = 25
theta = np.deg2rad(theta_deg)

# updated r0 from geometry
r0 = 0.005 + 0.06 * np.tan(theta)
z0 = z_top

# line
z_line = np.linspace(z_bottom, z_top, 200)
r_line = r0 + (z0 - z_line) * np.tan(theta)

# bottom intersection
r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

# region polygon
poly_r = [0, 0, r0, r_bottom, 0]
poly_z = [z_bottom, z_top, z_top, z_bottom, z_bottom]

fig, ax = plt.subplots(figsize=(7,7))

# shaded region
ax.fill(poly_r, poly_z, color='magenta', alpha=0.2)

# boundaries
ax.plot([0,0],[z_bottom,z_top],'m',lw=2)
ax.plot([0,r0],[z_top,z_top],'m',lw=2)
ax.plot([0,r_bottom],[z_bottom,z_bottom],'m',lw=2)

# slanted line
ax.plot(r_line, z_line, '--', color='peru', lw=2)

# mark point
ax.plot(r0, z0, 'bo')
ax.text(r0 + 0.001, z0 + 0.001, '$(r_0,z_0)$')

# angle arc
arc_r = r0 + 0.015*np.sin(np.linspace(0, theta, 50))
arc_z = z0 - 0.015*np.cos(np.linspace(0, theta, 50))
ax.plot(arc_r, arc_z, color='peru')

ax.text(r0 + 0.01, z0 - 0.01, r'$\theta=25^\circ$', color='peru')

# labels
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')
ax.set_title('Integration region geometry')
ax.set_aspect('equal')

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)

plt.show()

In [44]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# -----------------------------
# Geometry of the integration region
# -----------------------------
z_top = 0.80
z_bottom = 0.75

theta_deg = 25
theta = np.deg2rad(theta_deg)

r0 = 0.005 + 0.06 * np.tan(theta)
z0 = z_top

# -----------------------------
# Build coordinate arrays
# -----------------------------
# Assuming:
#   rr   -> 1D radial coordinate array in cm
#   zz   -> 1D axial coordinate array in cm
#   dens -> 2D density array with shape (len(zz), len(rr))
#
# Replace rr, zz by your actual coordinate arrays if their names differ.
rr = np.linspace(0, 1, dens.shape[0])  # example radial coordinates
zz = np.linspace(0, 1, dens.shape[1])  # example axial coordinates
R, Z = np.meshgrid(rr, zz)

# right boundary of the integration region
rmax = r0 + (z0 - Z) * np.tan(theta)

# mask
mask = (
    (Z >= z_bottom) &
    (Z <= z_top) &
    (R >= 0.0) &
    (R <= rmax)
)

# for plotting the slanted boundary
z_line = np.linspace(z_bottom, z_top, 300)
r_line = r0 + (z0 - z_line) * np.tan(theta)

r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

f1 = ax.imshow(
    np.log10(dens),
    cmap=newcmap,
    origin='lower',
    extent=[rr.min(), rr.max(), zz.min(), zz.max()],
    vmin=-6, vmax=1,
    aspect='equal',
)

# overlay the masked region
ax.contourf(
    R, Z, mask.astype(float),
    levels=[0.5, 1.5],
    colors=['magenta'],
    alpha=0.22
)

# draw boundaries of the integration region
ax.plot(r_line, z_line, '--', color='white', lw=2, label='integration boundary')
ax.plot([0, r0], [z_top, z_top], '-', color='white', lw=2)
ax.plot([0, r_bottom], [z_bottom, z_bottom], '-', color='white', lw=2)
ax.plot([0, 0], [z_bottom, z_top], '-', color='white', lw=2)

# optional: mark the top point
ax.plot(r0, z0, 'wo', ms=5)
ax.text(r0 + 0.002, z0 + 0.002, r'$(r_0,z_0)$', color='white')

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')
# ax.set_title('Time = {:.1f} ns'.format(time_now))

fig.colorbar(f1, cax=cax, orientation='vertical')

ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

In [46]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --------------------------------------------------
# Geometry of the integration region
# --------------------------------------------------
z_top = 0.803
z_bottom = 0.75

theta_deg = 25.0
theta = np.deg2rad(theta_deg)

r0 = 0.005 + 0.06 * np.tan(theta)
z0 = z_top

# --------------------------------------------------
# Coordinates
# --------------------------------------------------
# Assume:
#   rr   : 1D radial array in cm
#   zz   : 1D axial array in cm
#   dens : 2D array with shape (len(zz), len(rr))
#
# Example:
# R[i,j] = rr[j], Z[i,j] = zz[i]
R, Z = np.meshgrid(rr, zz)

# right boundary
rmax = r0 + (z0 - Z) * np.tan(theta)

# boolean mask
mask = (
    (Z >= z_bottom) &
    (Z <= z_top) &
    (R >= 0.0) &
    (R <= rmax)
)

# line for plotting
z_line = np.linspace(z_bottom, z_top, 400)
r_line = r0 + (z0 - z_line) * np.tan(theta)
r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

f1 = ax.imshow(
    np.log10(dens),
    cmap=newcmap,
    origin='lower',
    extent=[rr.min(), rr.max(), zz.min(), zz.max()],
    vmin=-6, vmax=1,
    aspect='equal',
)

# soft shading of the integration region
ax.contourf(
    R, Z, mask.astype(float),
    levels=[0.5, 1.5],
    colors=['magenta'],
    alpha=0.18
)

# sharp boundary
ax.plot(r_line, z_line, '--', color='white', lw=2.5, label='integration boundary')
ax.plot([0, r_bottom], [z_bottom, z_bottom], '-', color='white', lw=2.5)
ax.plot([0, r0], [z_top, z_top], '-', color='white', lw=2.5)
ax.plot(r0, z0, 'wo', ms=7)
ax.text(r0 + 0.002, z0 + 0.0015, r'$(r_0,z_0)$', color='white', fontsize=14)

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')

fig.colorbar(f1, cax=cax, orientation='vertical')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

# --------------------------------------------------
# Integration
# --------------------------------------------------
# Axisymmetric volume element:
# dV = 2*pi*r*dr*dz
#
# So if dens is a number density [cm^-3], the result is a particle number.
# If dens is a mass density [g/cm^3], the result is a mass [g].

dr = np.gradient(rr)          # handles nonuniform rr too
dz = np.gradient(zz)          # handles nonuniform zz too

DR, DZ = np.meshgrid(dr, dz)

dV = 2.0 * np.pi * R * DR * DZ

integrated_quantity = np.sum(dens[mask] * dV[mask])

print("r0 =", r0, "cm")
print("Integrated quantity inside mask =", integrated_quantity)

r0 = 0.032978459489299915 cm
Integrated quantity inside mask = 4.409585355922905e-06 code_mass/code_length**3


In [47]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --------------------------------------------------
# Geometry of the integration region
# --------------------------------------------------
z_top = 0.803
z_bottom = 0.75

theta_deg = 25.0
theta = np.deg2rad(theta_deg)

# angle measured from the vertical
r0 = 0.005 + 0.06 * np.tan(theta)
z0 = z_top

# --------------------------------------------------
# Coordinates
# --------------------------------------------------
# rr   : 1D radial array in cm
# zz   : 1D axial array in cm
# dens : 2D mass density array in g/cm^3
#
# IMPORTANT:
# dens must be the VANADIUM mass density if you want the Vanadium ion number directly.
# If dens is the total mass density, see note below.

R, Z = np.meshgrid(rr, zz)

# right boundary of the mask
rmax = r0 + (z0 - Z) * np.tan(theta)

# mask
mask = (
    (Z >= z_bottom) &
    (Z <= z_top) &
    (R >= 0.0) &
    (R <= rmax)
)

# lines for plotting
z_line = np.linspace(z_bottom, z_top, 400)
r_line = r0 + (z0 - z_line) * np.tan(theta)
r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots()
fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='10%', pad=0.05)

# plot uses log10, but integration will use dens itself
f1 = ax.imshow(
    np.log10(dens),
    cmap=newcmap,
    origin='lower',
    extent=[rr.min(), rr.max(), zz.min(), zz.max()],
    vmin=-6, vmax=1,
    aspect='equal',
)

# soft shading
ax.contourf(
    R, Z, mask.astype(float),
    levels=[0.5, 1.5],
    colors=['magenta'],
    alpha=0.18
)

# mask contour
ax.contour(
    R, Z, mask.astype(float),
    levels=[0.5],
    colors='magenta',
    linewidths=2.0
)

# boundary lines
ax.plot(r_line, z_line, '--', color='white', lw=2.5, label='integration boundary')
ax.plot([0, r_bottom], [z_bottom, z_bottom], '-', color='white', lw=2.5)
ax.plot([0, r0], [z_top, z_top], '-', color='white', lw=2.5)
ax.plot(r0, z0, 'wo', ms=7)
ax.text(r0 + 0.002, z0 + 0.001, r'$(r_0,z_0)$', color='white', fontsize=14)

ax.set_xlim(0, 0.1)
ax.set_ylim(0.72, 0.83)
ax.set_xlabel('r (cm)')
ax.set_ylabel('z (cm)')

fig.colorbar(f1, cax=cax, orientation='vertical')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

# --------------------------------------------------
# Integration: Vanadium ion number
# --------------------------------------------------
# dens is in g/cm^3
# Need number density:
#   n_V = rho_V / (A_V * m_u)
#
# A_V: atomic mass of Vanadium
# m_u: atomic mass unit in g

A_V = 50.9415
m_u = 1.66053906660e-24   # g

# If dens is already VANADIUM mass density:
rho_V = dens

n_V = rho_V / (A_V * m_u)   # cm^-3

# cell sizes
dr = np.gradient(rr)
dz = np.gradient(zz)

DR, DZ = np.meshgrid(dr, dz)

# axisymmetric cell volume
dV = 2.0 * np.pi * R * DR * DZ   # cm^3

# total Vanadium ion number
N_V = np.sum(n_V[mask] * dV[mask])

# optional: total Vanadium mass in the region
M_V = np.sum(rho_V[mask] * dV[mask])   # g

print(f"theta = {theta_deg:.1f} deg")
print(f"r0 = {r0:.6f} cm")
print(f"z_top = {z_top:.6f} cm")
print(f"z_bottom = {z_bottom:.6f} cm")
print(f"Total Vanadium mass in mask = {M_V:.6e} g")
print(f"Total Vanadium ion number in mask = {N_V:.6e}")

theta = 25.0 deg
r0 = 0.032978 cm
z_top = 0.803000 cm
z_bottom = 0.750000 cm
Total Vanadium mass in mask = 4.409585e-06 code_mass/code_length**3 g
Total Vanadium ion number in mask = 5.212870e+16 code_mass/code_length**3


In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import yt

def analyze_vanadium_region(
    time_index,
    z_top,
    z_bottom,
    filedir='/Users/yao/Documents/Data/LULI2000/neutron_Vincent/vincent_t5_2um/',
    prefix='GEKKO_hdf5_chk_',
    theta_deg=20.0,
    r_left=0.0,
    r_plot_max=0.10,
    z_plot_min=0.72,
    z_plot_max=0.83,
    log_threshold=-8.5,
    vmin=-6.0,
    vmax=1.0,
    cmap_name='inferno',
    A_V=50.9415,
    m_u=1.66053906660e-24,
    make_plot=True,
):
    """
    Load one FLASH checkpoint, plot Vanadium mass density, build the geometric mask,
    and compute integrated Vanadium mass and ion number.

    Parameters
    ----------
    time_index : int
        Output number, e.g. 0, 9, etc.
    z_top : float
        Upper z boundary of the integration region [cm].
    z_bottom : float
        Lower z boundary of the integration region [cm].

    Optional Parameters
    -------------------
    filedir : str
        Directory containing FLASH output files.
    prefix : str
        Filename prefix.
    theta_deg : float
        Cone angle measured from the vertical [deg].
    r_left : float
        Left radial boundary [cm], usually 0.
    r_plot_max : float
        Right plot limit in r [cm].
    z_plot_min, z_plot_max : float
        Plot limits in z [cm].
    log_threshold : float
        Values below this log10 threshold are shown as white.
    vmin, vmax : float
        Color scale limits for log10 plot.
    cmap_name : str
        Colormap name.
    A_V : float
        Vanadium atomic mass.
    m_u : float
        Atomic mass unit in grams.
    make_plot : bool
        Whether to generate the figure.

    Returns
    -------
    result : dict
        Dictionary containing arrays, mask, and integrated quantities.
    """

    # ------------------------------------------
    # File loading
    # ------------------------------------------
    filename = f"{prefix}{time_index:04d}"
    fullpath = os.path.join(filedir, filename)

    data_yt = yt.load(fullpath)
    data_yt_map = data_yt.covering_grid(
        level=0,
        left_edge=[0, 0.0, 0.0],
        dims=data_yt.domain_dimensions
    )

    # FLASH 2D cylindrical output read the same way you already do
    dens = data_yt_map['dens'][:, :, 0].T.d
    targ = data_yt_map['targ'][:, :, 0].T.d

    # ------------------------------------------
    # Coordinates
    # ------------------------------------------
    # These usually come from the domain edges
    dims = data_yt.domain_dimensions
    x_left, y_left, z_left = data_yt.domain_left_edge.d
    x_right, y_right, z_right = data_yt.domain_right_edge.d

    # Since you used [:,:,0].T, the plotted axes are:
    # horizontal = first FLASH axis after transpose
    # vertical   = second FLASH axis after transpose
    #
    # This matches your existing rr (radial) and zz (axial) style.
    rr = np.linspace(x_left, x_right, dims[0])
    zz = np.linspace(y_left, y_right, dims[1])

    # Safety check on shape
    if dens.shape != (len(zz), len(rr)):
        raise ValueError(
            f"Shape mismatch: dens.shape={dens.shape}, expected {(len(zz), len(rr))}. "
            "You may need to swap rr/zz or remove/add a transpose depending on your FLASH layout."
        )

    R, Z = np.meshgrid(rr, zz)

    # ------------------------------------------
    # Vanadium mass density
    # ------------------------------------------
    rho_V = dens * targ   # g/cm^3

    # ------------------------------------------
    # Geometry of the mask
    # ------------------------------------------
    theta = np.deg2rad(theta_deg)

    # angle measured from vertical
    r0 = 0.005 + 0.06 * np.tan(theta)
    z0 = z_top

    rmax = r0 + (z0 - Z) * np.tan(theta)

    mask = (
        (Z >= z_bottom) &
        (Z <= z_top) &
        (R >= r_left) &
        (R <= rmax)
    )

    # plotting lines
    z_line = np.linspace(z_bottom, z_top, 400)
    r_line = r0 + (z0 - z_line) * np.tan(theta)
    r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

    # ------------------------------------------
    # Integration
    # ------------------------------------------
    n_V = rho_V / (A_V * m_u)   # cm^-3

    dr = np.gradient(rr)
    dz = np.gradient(zz)
    DR, DZ = np.meshgrid(dr, dz)

    dV = 2.0 * np.pi * R * DR * DZ   # cm^3, axisymmetric volume element

    M_V = np.sum(rho_V[mask] * dV[mask])   # g
    N_V = np.sum(n_V[mask] * dV[mask])     # number of Vanadium ions

    # ------------------------------------------
    # Plot
    # ------------------------------------------
    fig = None
    ax = None

    if make_plot:
        # avoid log10(0)
        rho_V_safe = np.where(rho_V > 0, rho_V, np.nan)
        log_rho_V = np.log10(rho_V_safe)

        log_rho_V_masked = np.ma.masked_where(
            np.isnan(log_rho_V) | (log_rho_V < log_threshold),
            log_rho_V
        )

        cmap = plt.get_cmap(cmap_name).copy()
        cmap.set_bad(color='white')

        fig, ax = plt.subplots()
        fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='10%', pad=0.05)

        f1 = ax.imshow(
            log_rho_V_masked,
            # dens,
            cmap=cmap,
            origin='lower',
            extent=[rr.min(), rr.max(), zz.min(), zz.max()],
            vmin=vmin,
            vmax=vmax,
            aspect='equal',
        )

        # soft shading
        ax.contourf(
            R, Z, mask.astype(float),
            levels=[0.5, 1.5],
            colors=['magenta'],
            alpha=0.18
        )

        # mask contour
        # ax.contour(
        #     R, Z, mask.astype(float),
        #     levels=[0.5],
        #     colors='magenta',
        #     linewidths=2.0
        # )

        # boundary lines
        ax.plot(r_line, z_line, '--', color='blue', lw=2.0, label='integration boundary')
        ax.plot([r_left, r0], [z_top, z_top], '-', color='blue', lw=2.0)
        ax.plot([r_left, r_bottom], [z_bottom, z_bottom], '-', color='blue', lw=2.0)
        ax.plot(r0, z0, 'wo', ms=6)

        ax.set_xlim(0, r_plot_max)
        ax.set_ylim(z_plot_min, z_plot_max)
        ax.set_xlabel('r (cm)')
        ax.set_ylabel('z (cm)')
        ax.set_title(f'Output {time_index:04d}')

        cb = fig.colorbar(f1, cax=cax, orientation='vertical')
        cb.set_label(r'$\log_{10}(\rho_V\,[{\rm g/cm^3}])$')

        ax.legend(loc='upper right')
        fig.tight_layout()
        fig.savefig(filedir+f"vanadium_region_{time_index:04d}.png", dpi=300)
        plt.show()
        

    # ------------------------------------------
    # Return
    # ------------------------------------------
    result = {
        'time_index': time_index,
        'filename': filename,
        'rr': rr,
        'zz': zz,
        'dens': dens,
        'targ': targ,
        'rho_V': rho_V,
        'mask': mask,
        'r0': r0,
        'z_top': z_top,
        'z_bottom': z_bottom,
        'M_V_g': M_V,
        'N_V': N_V,
        'figure': fig,
        'ax': ax,
    }

    print(f"Loaded: {filename}")
    print(f"r0 = {r0:.6f} cm")
    print(f"z_top = {z_top:.6f} cm")
    print(f"z_bottom = {z_bottom:.6f} cm")
    print(f"Total Vanadium mass in mask = {M_V:.6e} g")
    print(f"Total Vanadium ion number in mask = {N_V:.6e}")

    return result

In [101]:
res9 = analyze_vanadium_region(
    time_index=9,
    z_top=0.803,
    z_bottom=0.743
)

yt : [INFO     ] 2026-03-13 11:52:28,253 Particle file found: GEKKO_hdf5_chk_0009
yt : [WARNING  ] 2026-03-13 11:52:28,257 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 11:52:28,275 Parameters: current_time              = 9.168184917484731e-10
yt : [INFO     ] 2026-03-13 11:52:28,276 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 11:52:28,277 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 11:52:28,277 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 11:52:28,278 Parameters: cosmological_simulation   = 0


Loaded: GEKKO_hdf5_chk_0009
r0 = 0.032978 cm
z_top = 0.803000 cm
z_bottom = 0.743000 cm
Total Vanadium mass in mask = 4.413152e-06 g
Total Vanadium ion number in mask = 5.217086e+16


In [100]:
res0 = analyze_vanadium_region(
    time_index=0,
    z_top=0.803,
    z_bottom=0.743
)

yt : [INFO     ] 2026-03-13 11:49:59,969 Particle file found: GEKKO_hdf5_chk_0000
yt : [WARNING  ] 2026-03-13 11:49:59,974 Extending theta dimension to 2PI + left edge.


yt : [INFO     ] 2026-03-13 11:49:59,995 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-13 11:49:59,995 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 11:49:59,996 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 11:49:59,996 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 11:49:59,997 Parameters: cosmological_simulation   = 0


Loaded: GEKKO_hdf5_chk_0000
r0 = 0.032978 cm
z_top = 0.803000 cm
z_bottom = 0.743000 cm
Total Vanadium mass in mask = 4.384824e-06 g
Total Vanadium ion number in mask = 5.183598e+16


In [8]:
def compare_two_times(t1, t2, z_top, z_bottom, **kwargs):
    res1 = analyze_vanadium_region(t1, z_top, z_bottom, **kwargs)
    res2 = analyze_vanadium_region(t2, z_top, z_bottom, **kwargs)

    print("\nComparison")
    print(f"N_V({t1}) = {res1['N_V']:.6e}")
    print(f"N_V({t2}) = {res2['N_V']:.6e}")
    print(f"Ratio     = {res2['N_V']/res1['N_V']:.6e}")
    print(f"Delta      = {res2['N_V']-res1['N_V']:.6e}")

    return res1, res2

In [9]:
res0, res9 = compare_two_times(
    0, 9,
    z_top=0.805,
    z_bottom=0.742,
)

yt : [INFO     ] 2026-03-13 13:49:42,701 Particle file found: GEKKO_hdf5_chk_0000
yt : [WARNING  ] 2026-03-13 13:49:42,708 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:49:42,731 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-13 13:49:42,732 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 13:49:42,732 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 13:49:42,732 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 13:49:42,733 Parameters: cosmological_simulation   = 0
yt : [INFO     ] 2026-03-13 13:49:47,272 Particle file found: GEKKO_hdf5_chk_0009
yt : [WARNING  ] 2026-03-13 13:49:47,279 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:49:47,296 Parameters: current_time              = 9.168184917484731e-10
yt : [INFO     ] 2026-03-13 13:49:47,297 Parameters: domain_dimensions         = [256

Loaded: GEKKO_hdf5_chk_0000
r0 = 0.026838 cm
z_top = 0.805000 cm
z_bottom = 0.742000 cm
Total Vanadium mass in mask = 3.093890e-06 g
Total Vanadium ion number in mask = 3.657497e+16
Loaded: GEKKO_hdf5_chk_0009
r0 = 0.026838 cm
z_top = 0.805000 cm
z_bottom = 0.742000 cm
Total Vanadium mass in mask = 3.052864e-06 g
Total Vanadium ion number in mask = 3.608998e+16

Comparison
N_V(0) = 3.657497e+16
N_V(9) = 3.608998e+16
Ratio     = 9.867398e-01
Delta      = -4.849927e+14


In [5]:
import numpy as np
import yt

def vanadium_axis_1D(
        time_index,
        z_top,
        z_bottom,
        filedir='/Users/yao/Documents/Data/LULI2000/neutron_Vincent/vincent_t5_2um/',
        prefix='GEKKO_hdf5_chk_',
        A_V=50.9415,
        m_u=1.66053906660e-24
        ):

    # -------------------------
    # Load FLASH checkpoint
    # -------------------------
    filename = f"{prefix}{time_index:04d}"
    data_yt = yt.load(filedir + filename)

    data_yt_map = data_yt.covering_grid(
        level=0,
        left_edge=[0,0,0],
        dims=data_yt.domain_dimensions
    )

    dens = data_yt_map['dens'][:,:,0].T.d
    targ = data_yt_map['targ'][:,:,0].T.d

    # -------------------------
    # Coordinates
    # -------------------------
    dims = data_yt.domain_dimensions
    left = data_yt.domain_left_edge.d
    right = data_yt.domain_right_edge.d

    rr = np.linspace(left[0], right[0], dims[0])
    zz = np.linspace(left[1], right[1], dims[1])

    # -------------------------
    # Vanadium density
    # -------------------------
    rho_V = dens * targ

    # -------------------------
    # axis index
    # -------------------------
    i_axis = np.argmin(np.abs(rr - 0.0))

    rho_axis = rho_V[:, i_axis]

    # -------------------------
    # z region
    # -------------------------
    zmask = (zz >= z_bottom) & (zz <= z_top)

    dz = np.gradient(zz)

    # -------------------------
    # 1D integrals
    # -------------------------
    M_1D = np.sum(rho_axis[zmask] * dz[zmask])

    N_1D = np.sum(
        rho_axis[zmask] / (A_V * m_u) * dz[zmask]
    )

    print(f"\nOutput {time_index:04d}")
    print(f"Axis Vanadium mass per area = {M_1D:.6e} g/cm^2")
    print(f"Axis Vanadium ion number per area = {N_1D:.6e} cm^-2")

    return M_1D, N_1D

In [6]:
z_top = 0.803
z_bottom = 0.75

M0, N0 = vanadium_axis_1D(0, z_top, z_bottom)
M9, N9 = vanadium_axis_1D(9, z_top, z_bottom)

print("\nRatio (t9 / t0)")
print("Mass ratio =", M9/M0)
print("Ion number ratio =", N9/N0)

yt : [INFO     ] 2026-03-13 13:31:53,770 Particle file found: GEKKO_hdf5_chk_0000
yt : [WARNING  ] 2026-03-13 13:31:53,777 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:31:53,804 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-13 13:31:53,805 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 13:31:53,805 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 13:31:53,806 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 13:31:53,806 Parameters: cosmological_simulation   = 0
yt : [INFO     ] 2026-03-13 13:31:57,764 Particle file found: GEKKO_hdf5_chk_0009
yt : [WARNING  ] 2026-03-13 13:31:57,773 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:31:57,818 Parameters: current_time              = 9.168184917484731e-10
yt : [INFO     ] 2026-03-13 13:31:57,818 Parameters: domain_dimensions         = [256


Output 0000
Axis Vanadium mass per area = 1.193826e-03 g/cm^2
Axis Vanadium ion number per area = 1.411302e+19 cm^-2

Output 0009
Axis Vanadium mass per area = 1.205971e-03 g/cm^2
Axis Vanadium ion number per area = 1.425660e+19 cm^-2

Ratio (t9 / t0)
Mass ratio = 1.0101733784891977
Ion number ratio = 1.0101733784891977


In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import yt

def analyze_vanadium_region(
    time_index,
    z_top,
    z_bottom,
    filedir='/Users/yao/Documents/Data/LULI2000/neutron_Vincent/vincent_t5_2um/',
    prefix='GEKKO_hdf5_chk_',
    theta_deg=20.0,
    r_left=0.0,
    r_plot_max=0.10,
    z_plot_min=0.72,
    z_plot_max=0.83,
    log_threshold=-8.5,
    vmin=-6.0,
    vmax=1.0,
    cmap_name='inferno',
    A_V=50.9415,
    m_u=1.66053906660e-24,
    make_plot=True,
):
    """
    Load one FLASH checkpoint, plot Vanadium mass density, build the geometric mask,
    and compute integrated Vanadium mass and ion number.

    Parameters
    ----------
    time_index : int
        Output number, e.g. 0, 9, etc.
    z_top : float
        Upper z boundary of the integration region [cm].
    z_bottom : float
        Lower z boundary of the integration region [cm].

    Optional Parameters
    -------------------
    filedir : str
        Directory containing FLASH output files.
    prefix : str
        Filename prefix.
    theta_deg : float
        Cone angle measured from the vertical [deg].
    r_left : float
        Left radial boundary [cm], usually 0.
    r_plot_max : float
        Right plot limit in r [cm].
    z_plot_min, z_plot_max : float
        Plot limits in z [cm].
    log_threshold : float
        Values below this log10 threshold are shown as white.
    vmin, vmax : float
        Color scale limits for log10 plot.
    cmap_name : str
        Colormap name.
    A_V : float
        Vanadium atomic mass.
    m_u : float
        Atomic mass unit in grams.
    make_plot : bool
        Whether to generate the figure.

    Returns
    -------
    result : dict
        Dictionary containing arrays, mask, and integrated quantities.
    """

    # ------------------------------------------
    # File loading
    # ------------------------------------------
    filename = f"{prefix}{time_index:04d}"
    fullpath = os.path.join(filedir, filename)

    data_yt = yt.load(fullpath)
    data_yt_map = data_yt.covering_grid(
        level=0,
        left_edge=[0, 0.0, 0.0],
        dims=data_yt.domain_dimensions
    )

    # FLASH 2D cylindrical output read the same way you already do
    dens = data_yt_map['dens'][:, :, 0].T.d
    targ = data_yt_map['targ'][:, :, 0].T.d

    # ------------------------------------------
    # Coordinates
    # ------------------------------------------
    # These usually come from the domain edges
    dims = data_yt.domain_dimensions
    x_left, y_left, z_left = data_yt.domain_left_edge.d
    x_right, y_right, z_right = data_yt.domain_right_edge.d

    # Since you used [:,:,0].T, the plotted axes are:
    # horizontal = first FLASH axis after transpose
    # vertical   = second FLASH axis after transpose
    #
    # This matches your existing rr (radial) and zz (axial) style.
    rr = np.linspace(x_left, x_right, dims[0])
    zz = np.linspace(y_left, y_right, dims[1])

    # Safety check on shape
    if dens.shape != (len(zz), len(rr)):
        raise ValueError(
            f"Shape mismatch: dens.shape={dens.shape}, expected {(len(zz), len(rr))}. "
            "You may need to swap rr/zz or remove/add a transpose depending on your FLASH layout."
        )

    R, Z = np.meshgrid(rr, zz)

    # ------------------------------------------
    # Vanadium mass density
    # ------------------------------------------
    rho_V = dens * targ   # g/cm^3

    # ------------------------------------------
    # Geometry of the mask
    # ------------------------------------------
    theta = np.deg2rad(theta_deg)

    # angle measured from vertical
    r0 = 0.005 + 0.06 * np.tan(theta)
    z0 = z_top

    rmax = r0 + (z0 - Z) * np.tan(theta)

    mask = (
        (Z >= z_bottom) &
        (Z <= z_top) &
        (R >= r_left) &
        (R <= rmax)
    )

    # plotting lines
    z_line = np.linspace(z_bottom, z_top, 400)
    r_line = r0 + (z0 - z_line) * np.tan(theta)
    r_bottom = r0 + (z0 - z_bottom) * np.tan(theta)

    # ------------------------------------------
    # Integration
    # ------------------------------------------
    n_V = rho_V / (A_V * m_u)   # cm^-3

    dr = np.gradient(rr)
    dz = np.gradient(zz)
    DR, DZ = np.meshgrid(dr, dz)

    dV = 2.0 * np.pi * R * DR * DZ   # cm^3, axisymmetric volume element

    M_V = np.sum(rho_V[mask] * dV[mask])   # g
    N_V = np.sum(n_V[mask] * dV[mask])     # number of Vanadium ions

    # ------------------------------------------
    # Plot
    # ------------------------------------------
    fig = None
    ax = None

    if make_plot:
        # avoid log10(0)
        rho_V_safe = np.where(rho_V > 0, rho_V, np.nan)
        log_rho_V = np.log10(rho_V_safe)

        log_rho_V_masked = np.ma.masked_where(
            np.isnan(log_rho_V) | (log_rho_V < log_threshold),
            log_rho_V
        )

        cmap = plt.get_cmap(cmap_name).copy()
        cmap.set_bad(color='white')

        fig, ax = plt.subplots()
        fig.subplots_adjust(left=.15, bottom=.16, right=.99, top=.97)

        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='10%', pad=0.05)

        f1 = ax.imshow(
            log_rho_V_masked,
            # dens,
            cmap=cmap,
            origin='lower',
            extent=[rr.min(), rr.max(), zz.min(), zz.max()],
            vmin=vmin,
            vmax=vmax,
            aspect='equal',
        )

        # # soft shading
        # ax.contourf(
        #     R, Z, mask.astype(float),
        #     levels=[0.5, 1.5],
        #     colors=['magenta'],
        #     alpha=0.18
        # )

        # mask contour
        # ax.contour(
        #     R, Z, mask.astype(float),
        #     levels=[0.5],
        #     colors='magenta',
        #     linewidths=2.0
        # )

        # # boundary lines
        # ax.plot(r_line, z_line, '--', color='blue', lw=2.0, label='integration boundary')
        # ax.plot([r_left, r0], [z_top, z_top], '-', color='blue', lw=2.0)
        # ax.plot([r_left, r_bottom], [z_bottom, z_bottom], '-', color='blue', lw=2.0)
        # ax.plot(r0, z0, 'wo', ms=6)

        ax.set_xlim(0, r_plot_max)
        ax.set_ylim(z_plot_min, z_plot_max)
        ax.set_xlabel('r (cm)')
        ax.set_ylabel('z (cm)')
        ax.set_title(f'Output {time_index:04d}')

        cb = fig.colorbar(f1, cax=cax, orientation='vertical')
        cb.set_label(r'$\log_{10}(\rho_V\,[{\rm g/cm^3}])$')

        # ax.legend(loc='upper right')
        fig.tight_layout()
        fig.savefig(filedir+f"vanadium_region_{time_index:04d}_ori.png", dpi=300)
        plt.show()
        

    # ------------------------------------------
    # Return
    # ------------------------------------------
    result = {
        'time_index': time_index,
        'filename': filename,
        'rr': rr,
        'zz': zz,
        'dens': dens,
        'targ': targ,
        'rho_V': rho_V,
        'mask': mask,
        'r0': r0,
        'z_top': z_top,
        'z_bottom': z_bottom,
        'M_V_g': M_V,
        'N_V': N_V,
        'figure': fig,
        'ax': ax,
    }

    print(f"Loaded: {filename}")
    print(f"r0 = {r0:.6f} cm")
    print(f"z_top = {z_top:.6f} cm")
    print(f"z_bottom = {z_bottom:.6f} cm")
    print(f"Total Vanadium mass in mask = {M_V:.6e} g")
    print(f"Total Vanadium ion number in mask = {N_V:.6e}")

    return result

In [13]:
res0, res9 = compare_two_times(
    0, 9,
    z_top=0.805,
    z_bottom=0.742,
)

yt : [INFO     ] 2026-03-13 13:53:54,802 Particle file found: GEKKO_hdf5_chk_0000


yt : [WARNING  ] 2026-03-13 13:53:54,808 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:53:54,829 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-13 13:53:54,830 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 13:53:54,830 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-13 13:53:54,831 Parameters: domain_right_edge         = [1.         1.         6.28318531]
yt : [INFO     ] 2026-03-13 13:53:54,831 Parameters: cosmological_simulation   = 0
yt : [INFO     ] 2026-03-13 13:53:58,822 Particle file found: GEKKO_hdf5_chk_0009
yt : [WARNING  ] 2026-03-13 13:53:58,827 Extending theta dimension to 2PI + left edge.
yt : [INFO     ] 2026-03-13 13:53:58,844 Parameters: current_time              = 9.168184917484731e-10
yt : [INFO     ] 2026-03-13 13:53:58,844 Parameters: domain_dimensions         = [2560 2560    1]
yt : [INFO     ] 2026-03-13 13:53:58,845 Parameters: domain_left_edge

Loaded: GEKKO_hdf5_chk_0000
r0 = 0.026838 cm
z_top = 0.805000 cm
z_bottom = 0.742000 cm
Total Vanadium mass in mask = 3.093890e-06 g
Total Vanadium ion number in mask = 3.657497e+16
Loaded: GEKKO_hdf5_chk_0009
r0 = 0.026838 cm
z_top = 0.805000 cm
z_bottom = 0.742000 cm
Total Vanadium mass in mask = 3.052864e-06 g
Total Vanadium ion number in mask = 3.608998e+16

Comparison
N_V(0) = 3.657497e+16
N_V(9) = 3.608998e+16
Ratio     = 9.867398e-01
Delta      = -4.849927e+14
